# Part 1: The Telegrapher's Equation & Signal Dispersion

## **1.1 Context & Pedagogical Goal**

**User Observation:**
> "I am struggling to convey to my students in IT213 the difference between the physical infrastructure (ISP cost) and the logical IP address (DHCP). I want to frame this explaining the 'sheer cost' of the fixed infrastructure versus the dynamic allocation of the logical address."

**The Analogy: "The Hotel vs. The Guest"**
* **The ISP is the Hotel (Infrastructure):** The ISP spent billions building the fiber, switches, and power plants. This is the "Fixed Channel." Like a hotel building, it cannot move.
* **The User is the Guest (IP Address):** DHCP is the valet key. You do not own the room (IP); you lease it. If every guest kept their key forever, the hotel would run out of rooms.

---

## **1.2 The Physics of Dispersion (Telegrapher's Equation)**

**User Observation:**
> "Help me build a schematic connecting the rate of transmission versus distance when pulse width becomes wider due to dispersion. Frame this using the Nyquist sampling theorem."

**Theory: The Diffusion Limit**
In the limit where Inductance ($L$) is negligible compared to Resistance ($R$) and Capacitance ($C$), the Telegrapher's Equation simplifies from a Wave Equation to a **Diffusion Equation**:

$$
\frac{\partial^2 V}{\partial x^2} = RC \frac{\partial V}{\partial t}
$$

**The Analytic Solution (Step Response)**
For a step input, the signal at distance $x$ is described by the Complementary Error Function (erfc). The "rise time" degrades with the square of the distance:

$$
V(x,t) = V_0 \text{erfc}\left( \sqrt{\frac{RC x^2}{4t}} \right)
$$

**Frequency Domain (The Skin Effect)**
**User Observation:**
> "Modify the transfer function to include the attenuation factor (Q-factor) and skin depth as a function of frequency."

In the frequency domain, the transfer function $H(\omega)$ for a lossy line dominated by the Skin Effect exhibits attenuation that scales with $\sqrt{\omega}$. This acts as a heavy Low-Pass filter:

$$
H(x, \omega) = e^{-\gamma x} \quad \text{where} \quad \gamma \approx \sqrt{i \omega RC} = (1+i)\sqrt{\frac{\omega RC}{2}}
$$

* **Attenuation ($\alpha$):** $e^{-\sqrt{\omega}}$ (High frequencies die out faster)
* **Phase ($\beta$):** $e^{-i\sqrt{\omega}}$ (High frequencies arrive at different times, causing dispersion/ISI)

---

## **1.3 Implementation & Corrections**

**User Observation:**
> "The code contains one error. The signal is `fftshifted`... either the frequency bins or the signal bins have to be shifted."

**Correction:**
We removed the `fftshift` from the calculation pipeline. The `fft` and `ifft` functions in NumPy expect the DC component at index 0. Shifting is only used for visualization, not calculation.

**User Observation:**
> "There seems to be aliasing when the frequency domain final response is converted back to time domain... Pad the spectrum to avoid time aliasing."

**Correction (Pad-and-Crop):**
We implemented a **Zero-Padding** strategy. By padding the time-domain signal with zeros (doubling its length) before the FFT, we convert the circular convolution (inherent to FFT) into a linear convolution. This gives the dispersed "tail" room to expand without wrapping around to the beginning of the signal.

---

**References:**
1.  **Feynman, R. P.** (1964). *The Feynman Lectures on Physics*, Vol. II, Chapter 24-4: "The Transmission Line".
2.  **Pozar, D. M.** (2011). *Microwave Engineering* (4th ed.), Section 2.3: "The Terminated Lossless Transmission Line".
3.  **Farlow, S. J.** (1993). *Partial Differential Equations for Scientists and Engineers*, Lesson 18.

The Textbook Reference (Analytic Solution)

the Distortionless Line vs. Diffusion Limit.

Reference: Microwave Engineering, David M. Pozar (Standard EE text). Chapter: 2.3 "The Terminated Lossless Transmission Line" (General case).

The Solution Kernel (Frequency Domain): For a conductor where R≫ωL (the Low Frequency/Diffusion limit you are interested in), the propagation constant γ approximates to:
γ≈jωRC​=2ωRC​​(1+j)

This yields the specific Transfer Function form:
V(x)=V0​e−αxe−jβx
Where α(ω)=β(ω)≈2ωRC​​

This ω​ dependency is exactly what creates the "heavy tail" dispersion seen in the simulation, distinct from a simple RC exponential decay.

# The Telegrapher's Equation: Diffusion Limit

## 1. Physical Context
In the limit where Inductance ($L$) is negligible compared to Resistance ($R$) and Capacitance ($C$), the Telegrapher's Equation simplifies from a Wave Equation to a **Diffusion Equation**. This is characteristic of long coaxial cables or transmission lines where signal "smearing" (Dispersion) dominates over propagation delay.

## 2. The Mathematical Model
The voltage $V(x,t)$ evolves according to:
$$
\frac{\partial^2 V}{\partial x^2} = RC \frac{\partial V}{\partial t}
$$

### The Frequency Domain Solution
Transforming to the frequency domain, the solution for a pulse traveling distance $d$ is given by the Transfer Function $H(\omega)$:

$$
V(d, \omega) = V(0, \omega) \cdot e^{-\gamma(\omega)d}
$$

Where the propagation constant $\gamma$ in the diffusion limit is:
$$
\gamma(\omega) = \sqrt{j \omega R C} = \sqrt{\frac{\omega RC}{2}}(1 + j)
$$

This leads to two distinct effects:
1.  **Attenuation ($\alpha$):** The amplitude decays as $e^{-\sqrt{\omega}}$. Higher frequencies decay much faster, rounding the sharp corners of square waves.
2.  **Dispersion ($\beta$):** Different frequencies travel at different speeds, causing the pulse to spread out (Inter-Symbol Interference).

## 3. Nyquist & Sampling
To recover the signal without error, the symbol rate must be low enough that the "tail" of one dispersed pulse does not overlap significantly with the next. This relates to the **Nyquist Inter-Symbol Interference (ISI)** criterion.

Telegrapher's Equation in the limit where R (resistance) and C (capacitance) dominate over L (inductance) is mathematically identical to the Diffusion Equation (or Heat Equation).

When L→0, the Telegrapher's equation becomes:
∂x2∂2V​=RC∂t∂V​
The Solution Kernel (Impulse Response)

This is likely the "compact definition" you are looking for. The smearing of a sharp pulse (Delta function) as it travels distance x is given by the Heat Kernel.

Textbook Reference:

    Book: The Feynman Lectures on Physics, Vol. II. Chapter: 24-4, "The Transmission Line" (specifically the section on the "Diffusion" limit). Alternative: Partial Differential Equations for Scientists and Engineers by S.J. Farlow, Lesson 18.

The Analytic "Box" Solution: If you input a Delta pulse δ(t) at x=0, the voltage at distance x is:
h(x,t)=4πt3​xRC​​exp(−4tRCx2​)

Frequency Domain (Transfer Function): This is the LPF form you were asking for.
H(x,ω)=e−iωRC​⋅x

    Note the ω​: This is why it is distinct from a simple standard filter. The attenuation α(ω) scales with ω​. This is the "Skin Effect" law, characteristic of diffusive media.

    Same kernel used in electromagnetic depth sounding or pressure diffusion in porous media. The "distance" slider in the code effectively increases the RCx2 term, widening the Gaussian bell curve until the pulses overlap (ISI).

This is the most rigorous version. Since you are a specialist in numerical methods, we will switch from the Time Domain (ODE/IIR Filter) to the Frequency Domain (FFT/Convolution).

This allows us to implement the exact analytic solution for the Skin Effect, where attenuation α(ω) scales with ω​. This shape is notoriously difficult to simulate with standard time-domain ODE solvers (which prefer integer derivatives) but is trivial in the frequency domain.
The Physics: Skin Depth & Q

We are modeling the cable where the Skin Depth δ(ω) limits the penetration of the signal into the conductor.

    Skin Depth: δ(ω)=ωμσ2​​

        δ shrinks as frequency ω increases.

    Attenuation: α(ω)≈δ(ω)1​∝ω​

        This confirms your intuition: Higher frequencies see more "resistance" per meter.

    The "Q" Connection: If we view the cable as a resonator, the Quality Factor Q is related to the ratio of energy stored to energy lost per cycle. In transmission lines, Q≈2αβ​. Since β∝ω and α∝ω​, the "effective Q" of a copper cable actually changes with frequency.

The Code (FFT-Based Transmission Line)

This script performs the following:

    Takes the pulse train into the Frequency Domain (FFT).

    Applies the exact Transfer Function H(ω)=e−α(ω)de−jβ(ω)d.

    Returns to the Time Domain (IFFT).

In [ ]:
# @title A.1) Time Domain: Telegraph Dispersion
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import ipywidgets as widgets
from ipywidgets import interact

def plot_telegraph_time(distance_km=2.0, r_per_km=10.0, c_per_km=5.0):
    # Setup Time Axis
    fs = 1000
    duration = 2.0
    t = np.linspace(0, duration, int(fs * duration), endpoint=False)

    # Generate Morse "I T" (.. -)
    sig = np.zeros_like(t)
    dot = 0.08
    # Dots
    sig[(t >= 0.2) & (t < 0.2 + dot)] = 5
    sig[(t >= 0.2 + 2*dot) & (t < 0.2 + 3*dot)] = 5
    # Dash
    sig[(t >= 0.2 + 5*dot) & (t < 0.2 + 8*dot)] = 5

    # Physics: 1st Order RC Filter approx for Diffusion
    # Time constant tau ~ R * C * dist^2
    r_total = max(r_per_km * distance_km, 0.1)
    c_total = max(c_per_km * distance_km, 0.1)

    # In diffusion limit, tau scales with length squared
    tau = r_total * c_total

    # Filter Design
    # We map 'tau' to a cutoff frequency for the filter
    fc = 2000 / (tau * 50) # Scaling constant for visualization
    w_nyq = fc / (fs / 2)
    b, a = signal.butter(1, min(w_nyq, 0.99), btype='low')

    # Filter
    sig_rx = signal.lfilter(b, a, sig)

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(t, sig, 'k--', alpha=0.3, label='Sender')
    plt.plot(t, sig_rx, 'r-', linewidth=2, label='Receiver (Dispersed)')
    plt.axhline(2.5, color='gray', linestyle=':', label='Threshold')
    plt.title(f"Telegraph Diffusion (RC ~ Distance²)\nDistance: {distance_km}km | Time Constant $\\tau$: {tau:.2f}")
    plt.xlabel("Time (s)")
    plt.ylabel("Voltage (V)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Create Interactive Widget
interact(plot_telegraph_time,
         distance_km=(0.1, 10.0, 0.1),
         r_per_km=(1.0, 50.0, 1.0),
         c_per_km=(1.0, 50.0, 1.0));

In [ ]:
# @title A.2) Frequency Domain: Skin Effect & Dispersion
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft, ifft, fftfreq
from scipy.ndimage import gaussian_filter1d
import ipywidgets as widgets
from ipywidgets import interact

def plot_telegraph_freq(dist_km=2.0, resistivity=5.0, q_factor=1.0):
    fs = 2000
    duration = 2.0
    N = int(fs * duration)
    t = np.linspace(0, duration, N, endpoint=False)

    # Signal Generation
    sig = np.zeros_like(t)
    dot = 0.08
    sig[(t >= 0.2) & (t < 0.2 + dot)] = 5
    sig[(t >= 0.2 + 2*dot) & (t < 0.2 + 3*dot)] = 5
    sig[(t >= 0.2 + 5*dot) & (t < 0.2 + 8*dot)] = 5
    # Smooth corners to reduce infinite bandwidth (Anti-aliasing)
    sig = gaussian_filter1d(sig, sigma=3)

    # --- PADDING (Fixes Circular Convolution Aliasing) ---
    pad_len = N
    sig_padded = np.pad(sig, (0, pad_len), 'constant')

    # FFT
    freqs = fftfreq(len(sig_padded), 1/fs)
    omega = 2 * np.pi * freqs
    w_abs = np.abs(omega)

    # Transfer Function H(w) = exp(-gamma * x)
    # alpha ~ sqrt(w) (Skin Effect)
    alpha = (0.1 * np.sqrt(w_abs) * np.sqrt(resistivity)) / max(q_factor, 0.1)
    beta  = w_abs * 0.05 # Linear delay

    H = np.exp(-alpha * dist_km) * np.exp(-1j * beta * dist_km)

    # Apply & IFFT
    sig_rx_padded = np.real(ifft(fft(sig_padded) * H))

    # --- CROP PADDING ---
    sig_rx = sig_rx_padded[:N]

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Time Domain
    ax1.plot(t, sig, 'k--', alpha=0.3, label='Sender')
    ax1.plot(t, sig_rx, 'r-', linewidth=2, label='Receiver')
    ax1.set_title(f"Time Domain: Linear Convolution (No Aliasing)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    # Frequency Domain (Positive only)
    mask = freqs[:len(freqs)//2] > 0
    f_pos = freqs[:len(freqs)//2][mask]
    H_mag = np.abs(H[:len(freqs)//2][mask])

    ax2.plot(f_pos, H_mag, 'b-', linewidth=2)
    ax2.set_xlim(0, 100)
    ax2.set_title("Cable Frequency Response $|H(\\omega)| = e^{-\\sqrt{\\omega}}$")
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("Attenuation")
    ax2.grid(True, alpha=0.3)

    plt.show()

interact(plot_telegraph_freq,
         dist_km=(0.1, 10.0, 0.1),
         resistivity=(0.1, 20.0, 0.5),
         q_factor=(0.1, 5.0, 0.1));

# Part 1: The Telegrapher's Equation & Signal Dispersion

## **1.1 Context: Infrastructure vs. Logical Addressing**

**User Observation:**
> "I am struggling to convey to my students in IT213 the difference between the physical infrastructure (ISP cost) and the logical IP address (DHCP). I want to frame this explaining the 'sheer cost' of the fixed infrastructure versus the dynamic allocation of the logical address."

**The Analogy: "The Hotel vs. The Guest"**
The concept is abstract because users see the cable plugged in 24/7 and assume the "connection" is permanent. You are dealing with two different layers here: **The Physical Road** (Infrastructure) and **The Street Address** (IP).

* **The ISP is the Hotel (Infrastructure):** The ISP spent billions building the fiber, switches, and power plants. This is the "Fixed Channel." Like a hotel building, it cannot move. The infrastructure represents a massive capital expenditure (CapEx).
* **The User is the Guest (IP Address):** DHCP is the valet key. You do not own the room (IP); you lease it. If every guest kept their room key forever, the hotel would run out of capacity. DHCP allows the ISP to recycle the scarce resource (IPv4 addresses) among the transient population of users.

---

## **1.2 The Physics of Dispersion**

**User Observation:**
> "Help me build a schematic connecting the rate of transmission versus distance when pulse width becomes wider due to dispersion. Frame this using the Nyquist sampling theorem."

**Theory: The Diffusion Limit**
We bridge the gap between "Physics" (the cable blurs the signal) and "Logic" (the receiver can't read it). In the limit where Inductance ($L$) is negligible compared to Resistance ($R$) and Capacitance ($C$), the Telegrapher's Equation simplifies from a Wave Equation to a **Diffusion Equation**:

$$
\frac{\partial^2 V}{\partial x^2} = RC \frac{\partial V}{\partial t}
$$

**The Analytic Solution (Step Response)**
For a step input, the signal at distance $x$ is described by the Complementary Error Function (erfc). The "rise time" degrades with the square of the distance, meaning doubling the distance quadruples the signal blur:

$$
V(x,t) = V_0 \text{erfc}\left( \sqrt{\frac{RC x^2}{4t}} \right)
$$

**User Observation:**
> "Modify the transfer function to include the attenuation factor (Q-factor) and skin depth as a function of frequency."

**Frequency Domain (The Skin Effect)**
In the frequency domain, the transfer function $H(\omega)$ for a lossy line dominated by the Skin Effect exhibits attenuation that scales with $\sqrt{\omega}$. This acts as a heavy Low-Pass filter:

$$
H(x, \omega) = e^{-\gamma x} \quad \text{where} \quad \gamma \approx \sqrt{i \omega RC} = (1+i)\sqrt{\frac{\omega RC}{2}}
$$

* **Attenuation ($\alpha$):** $e^{-\sqrt{\omega}}$ (High frequencies die out faster).
* **Phase ($\beta$):** $e^{-i\sqrt{\omega}}$ (High frequencies arrive at different times, causing dispersion/ISI).

---

## **1.3 Signal Processing Corrections**

**User Observation:**
> "The code contains one error. The signal is `fftshifted`... either the frequency bins or the signal bins have to be shifted."

**Correction:**
We removed the `fftshift` from the calculation pipeline. The `fft` and `ifft` functions in NumPy expect the DC component at index 0. Shifting is only used for visualization, not calculation.

**User Observation:**
> "There seems to be aliasing when the frequency domain final response is converted back to time domain... Pad the spectrum to avoid time aliasing."

**Correction (Pad-and-Crop):**
We implemented a **Zero-Padding** strategy. By padding the time-domain signal with zeros (doubling its length) before the FFT, we convert the circular convolution (inherent to FFT) into a linear convolution. This gives the dispersed "tail" room to expand without wrapping around to the beginning of the signal.

**References (Part 1):**
1.  **Feynman, R. P.** (1964). *The Feynman Lectures on Physics*, Vol. II, Chapter 24-4: "The Transmission Line".
2.  **Pozar, D. M.** (2011). *Microwave Engineering* (4th ed.), Section 2.3: "The Terminated Lossless Transmission Line".
3.  **Farlow, S. J.** (1993). *Partial Differential Equations for Scientists and Engineers*, Lesson 18.

---

# Part 1: The Telegrapher's Equation & Signal Dispersion

## **1.1 Context: Infrastructure vs. Logical Addressing**

**User Observation:**
> "I am struggling to convey to my students in IT213 at CUE the difference and the reason why we use as end user DHCP and that ISPs have fixed IP. I want to frame this question explaining that the ISP have a sheer cost since they have infrastructure. Help me create tiny glimpse of this contrasts to make it evident to my students that this is the reason why we use DHCP."

**The Analogy: "The Hotel vs. The Guest"**
The concept is abstract because users see the cable plugged in 24/7 and assume the "connection" is permanent. You are dealing with two different layers here: **The Physical Road** (Infrastructure) and **The Street Address** (IP).

* **The ISP is the Hotel (Fixed IP / Infrastructure):** The ISP spent billions building the fiber, switches, and power plants. This is the "Fixed Channel." Like a hotel building, it cannot move. The infrastructure represents a massive capital expenditure (CapEx) and requires a permanent, reachable address (Static IP) so the network can find it.
* **The End User is the Guest (Dynamic IP / DHCP):** DHCP is the valet key. You do not own the room (IP); you lease it. If every guest kept their room key forever, the hotel would run out of capacity. DHCP allows the ISP to recycle the scarce resource (IPv4 addresses) among the transient population of users.

---

## **1.2 The Physics of Dispersion**

**User Observation:**
> "Help me build a schematics connecting the Telegraph rate of transmission problem versus distance when pulsewidth becomes wider as it suffers dispersion through the cable and how Nyquist sampling theorem allowed to frame this question on in term of the sampling rate."

**Theory: The Diffusion Limit**
We bridge the gap between "Physics" (the cable blurs the signal) and "Logic" (the receiver can't read it). In the limit where Inductance ($L$) is negligible compared to Resistance ($R$) and Capacitance ($C$), the Telegrapher's Equation simplifies from a Wave Equation to a **Diffusion Equation**:

$$
\frac{\partial^2 V}{\partial x^2} = RC \frac{\partial V}{\partial t}
$$

**The Analytic Solution (Step Response)**
For a step input, the signal at distance $x$ is described by the Complementary Error Function (erfc). The "rise time" degrades with the square of the distance, meaning doubling the distance quadruples the signal blur:

$$
V(x,t) = V_0 \text{erfc}\left( \sqrt{\frac{RC x^2}{4t}} \right)
$$

**User Observation:**
> "Let's elaborate third version... keep analytic solution the SAME but the attenuation factor and skin depth to be a function of the frequency... So modify transfer function to include attenuation factor (Q-factor)."

**Frequency Domain (The Skin Effect)**
In the frequency domain, the transfer function $H(\omega)$ for a lossy line dominated by the Skin Effect exhibits attenuation that scales with $\sqrt{\omega}$. This acts as a heavy Low-Pass filter:

$$
H(x, \omega) = e^{-\gamma x} \quad \text{where} \quad \gamma \approx \sqrt{i \omega RC} = (1+i)\sqrt{\frac{\omega RC}{2}}
$$

* **Attenuation ($\alpha$):** $e^{-\sqrt{\omega}}$ (High frequencies die out faster).
* **Phase ($\beta$):** $e^{-i\sqrt{\omega}}$ (High frequencies arrive at different times, causing dispersion/ISI).

---

## **1.3 Signal Processing Corrections**

**User Observation:**
> "The code contains one error. The signal is fftshifted when transformed into frequency domain so either the frequency bins designed for the cable transfer function or the signal bins have to be fftshfited."

**Correction:**
We removed the `fftshift` from the calculation pipeline. The `fft` and `ifft` functions in NumPy expect the DC component at index 0. Shifting is only used for visualization, not calculation.

**User Observation:**
> "It is fantastic, however there seems to be aliasing when the frequency domain final response is converted back to time domain... Pad the spectrum to avoid time aliasing."

**Correction (Pad-and-Crop):**
We implemented a **Zero-Padding** strategy. By padding the time-domain signal with zeros (doubling its length) before the FFT, we convert the circular convolution (inherent to FFT) into a linear convolution. This gives the dispersed "tail" room to expand without wrapping around to the beginning of the signal.

**References (Part 1):**
1.  **Feynman, R. P.** (1964). *The Feynman Lectures on Physics*, Vol. II, Chapter 24-4: "The Transmission Line".
2.  **Pozar, D. M.** (2011). *Microwave Engineering* (4th ed.), Section 2.3: "The Terminated Lossless Transmission Line".
3.  **Farlow, S. J.** (1993). *Partial Differential Equations for Scientists and Engineers*, Lesson 18.

---

# Part 2: 2D Wavefront Propagation & Illumination

## **2.1 Time-Domain Wave Propagation (FDTD)**

**User Observation:**
> "I want to include some signal processing examples like this cable attenuation due to Q, and a forward model for (amplitude part only) of a wavefront propagation simulation for them to visualize what is going on. It solves Wave Operator $L=[D_{xx} + D_{yy}] + \omega^2/C^2(x,y)$."

**Theory: The 2D Acoustic Wave Equation**
We simulate the propagation of the WiFi signal using the scalar wave equation solved via the Finite-Difference Time-Domain (FDTD) method.

**The Equation:**
$$
\frac{\partial^2 u}{\partial t^2} = c(x,z)^2 \nabla^2 u
$$

**Numerical Discretization (Stencil):**
We use a standard second-order central difference approximation:

$$
u_{i,j}^{n+1} = 2u_{i,j}^n - u_{i,j}^{n-1} + \left( \frac{c \Delta t}{\Delta x} \right)^2 \left[ u_{i+1,j}^n + u_{i-1,j}^n + u_{i,j+1}^n + u_{i,j-1}^n - 4u_{i,j}^n \right]
$$

**User Observation (Stability):**
> "There is dispersion in the mesh, it seems the Courant condition was violated... Since you used FDTD solver I am assuming that being a time marching method there is no way to add regularization to the solver to avoid instability right?"

**Correction (Regularization):**
We *can* add regularization. We implemented **Artificial Viscosity** (smoothing) in the update step to suppress high-frequency checkerboard noise. This acts as a low-pass filter in space:

$$
u_{i,j}^{n+1} \leftarrow (1-\eta)u_{i,j}^{n+1} + \eta (\text{Average of Neighbors})
$$

**User Observation (Wrap-around):**
> "We have the same issue we had with the spectrum in the telegraph equation example there is wraparound in time for the wavefront it appears reflecting before being propagated. So add padding and then discard padding after."

**Correction (Padded Domain):**
We implemented a **Padded Solver**. A hidden "Sponge Layer" (padding) is added around the visible room. Waves entering this zone are damped ($ \times 0.9$) and die before reaching the periodic boundary.

---

## **2.2 Illumination Pattern (Steady-State Coverage)**

**User Observation:**
> "You forgot to add the amplitude envelope illumination code to show the illumination pattern. This can be done solving instead of Helmholtz, approximating the problem as a diffusion equation problem."

**Theory: Signal Amplitude as Diffusion**
To visualize the steady-state signal strength (coverage map) without solving the computationally expensive Helmholtz equation, we approximate the signal envelope $A(x,z)$ using the **Steady-State Diffusion Equation** with decay. This treats the antenna as a heat source and the concrete column as an insulator.

**The Equation (Modified Fick's Law):**
$$
-D(x,z) \nabla^2 A(x,z) + \gamma A(x,z) = S(x,z)
$$

* $A(x,z)$: Signal Amplitude.
* $D(x,z)$: Diffusivity (High in air, Low in concrete).
* $\gamma$: Path loss / Decay factor.
* $S(x,z)$: Source term.

**User Observation (Stiffness):**
> "The constant for diffusion you defined seems to make the system stiff... Can this be done via an implicit upwind method? can you use a similar solver then as in the FDTD propagation problem and discard leftovers form each iteration to avoid RAM memory saturation?"

**Correction (Direct Solver):**
Time-stepping diffusion problems to steady-state is computationally inefficient and prone to stiffness (static results) or ADI artifacts (diagonal tearing). We replaced the iterative solver with a **Direct Sparse Solver** (`scipy.sparse.linalg.spsolve`). This solves the linear system $\mathbf{M}\mathbf{x}=\mathbf{b}$ in one shot, guaranteeing a stable solution without artifacts or convergence issues.

**References (Part 2):**
1.  **Taflove, A. & Hagness, S. C.** (2005). *Computational Electrodynamics: The Finite-Difference Time-Domain Method*. Artech House.
2.  **Durgin, G.** (2003). *Space-Time Wireless Channels*. Prentice Hall, Chapter 4: "Diffuse Scattering".
3.  **Peaceman, D. W., & Rachford, H. H.** (1955). "The numerical solution of parabolic and elliptic differential equations." *J. Soc. Ind. Appl. Math*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from numpy.fft import fft, ifft, fftfreq
from scipy.ndimage import gaussian_filter1d

# --- 1. Signal Generation ---
fs = 2000  # Sample rate (Hz)
duration = 2.0
N = int(fs * duration)
t = np.linspace(0, duration, N, endpoint=False)

def generate_smooth_morse(t):
    dot_len = 0.08
    sig = np.zeros_like(t)

    # "I T" pattern (.. -)
    sig[(t >= 0.2) & (t < 0.2 + dot_len)] = 5
    sig[(t >= 0.2 + 2*dot_len) & (t < 0.2 + 3*dot_len)] = 5
    dash_start = 0.2 + 5*dot_len
    sig[(t >= dash_start) & (t < dash_start + 3*dot_len)] = 5

    # Anti-aliasing (Slew rate limit)
    sig_smooth = gaussian_filter1d(sig, sigma=5)
    return sig_smooth

sig_sender = generate_smooth_morse(t)

# --- 2. PADDED Frequency Domain Logic ---
# To avoid Time-Aliasing (Circular Wrap-around), we must pad the input.
# We double the length (N_pad = 2*N) to approximate Linear Convolution.
pad_amount = N
N_padded = N + pad_amount

# Pre-calculate frequency axis for the PADDED length
# The freq resolution increases (df gets smaller), but max freq (Nyquist) stays same.
freqs_padded = fftfreq(N_padded, 1/fs)
omega_padded = 2 * np.pi * freqs_padded
pos_mask_padded = freqs_padded >= 0

def calculate_response(sig_time, dist_km, resistivity, q_factor):
    # 1. Zero-Pad the Time Domain Signal
    # We add zeros to the end so the dispersion tail has somewhere to go
    sig_padded = np.pad(sig_time, (0, pad_amount), 'constant')

    # 2. FFT of the PADDED signal
    sig_f = fft(sig_padded)

    # 3. Compute Transfer Function H(w) on the PADDED grid
    w_abs = np.abs(omega_padded)

    # Skin Effect: Alpha ~ sqrt(w)
    alpha = (0.1 * np.sqrt(w_abs) * np.sqrt(resistivity)) / max(q_factor, 0.1)

    # Linear Phase Shift (Time Delay)
    beta = w_abs * 0.05

    # Transfer Function
    H = np.exp(-alpha * dist_km) * np.exp(-1j * beta * dist_km)

    # 4. Apply Filter
    sig_rx_f = sig_f * H

    # 5. IFFT to Time Domain
    sig_rx_padded = np.real(ifft(sig_rx_f))

    # 6. Truncate / Crop
    # We chop off the padding to match the original plot axis.
    # The "tail" that would have wrapped is now safely in the discarded padded region.
    sig_rx = sig_rx_padded[:N]

    return sig_rx, H, sig_rx_padded

# --- 3. Visualization Setup ---
fig, (ax_time, ax_freq) = plt.subplots(2, 1, figsize=(10, 8))
plt.subplots_adjust(bottom=0.30, hspace=0.4)

# Initial Parameters
init_d = 2.0
init_rho = 5.0
init_q = 1.0

# Calculate Initial State
y_init, H_init, _ = calculate_response(sig_sender, init_d, init_rho, init_q)

# --- SUBPLOT 1: TIME DOMAIN ---
l_sender_t, = ax_time.plot(t, sig_sender, 'k--', alpha=0.3, label='Sender')
l_receiver_t, = ax_time.plot(t, y_init, 'r-', linewidth=1.5, label='Receiver')

ax_time.set_title("Time Domain: Linear Convolution (Zero-Padded)", fontsize=12)
ax_time.set_ylabel("Voltage (V)")
ax_time.set_ylim(-1, 6)
ax_time.legend(loc='upper right')
ax_time.grid(True, alpha=0.3)
ax_time.axhline(2.5, color='gray', linestyle=':', label='Threshold')

# --- SUBPLOT 2: FREQUENCY DOMAIN ---
# Plot |H(f)|
freqs_view = freqs_padded[pos_mask_padded]
ax_freq_h = ax_freq.twinx()
# We take the H_init corresponding to the padded calculation
l_H_mag, = ax_freq_h.plot(freqs_view, np.abs(H_init[pos_mask_padded]), 'b-', linewidth=2, alpha=0.6, label='Cable |H(f)|')
ax_freq_h.set_ylabel("|H(f)|", color='blue')
ax_freq_h.set_ylim(0, 1.1)

# Plot Spectrum
# Use padded spectrum for higher resolution view
sender_spec = np.abs(fft(np.pad(sig_sender, (0, pad_amount))))[pos_mask_padded]
sender_spec = sender_spec / np.max(sender_spec)
l_spec, = ax_freq.plot(freqs_view, sender_spec, 'k-', alpha=0.2, label='Input Spectrum')

ax_freq.set_title("Frequency Domain: Cable Response", fontsize=12)
ax_freq.set_xlabel("Frequency (Hz)")
ax_freq.set_xlim(0, 100)
ax_freq.set_ylabel("Norm. Amplitude")
ax_freq.grid(True, alpha=0.3)

# --- Sliders ---
ax_dist = plt.axes([0.2, 0.18, 0.6, 0.03], facecolor='lightgray')
s_dist = Slider(ax_dist, 'Distance (km)', 0.1, 10.0, valinit=init_d, color='red')

ax_rho = plt.axes([0.2, 0.13, 0.6, 0.03], facecolor='lightgray')
s_rho = Slider(ax_rho, 'Resistivity', 0.1, 20.0, valinit=init_rho, color='blue')

ax_q = plt.axes([0.2, 0.08, 0.6, 0.03], facecolor='lightgray')
s_q = Slider(ax_q, 'Quality Factor', 0.1, 5.0, valinit=init_q, color='green')

def update(val):
    d = s_dist.val
    rho = s_rho.val
    q = s_q.val

    # Recalculate
    sig_rx, H, _ = calculate_response(sig_sender, d, rho, q)

    # Update Time Plot
    l_receiver_t.set_ydata(sig_rx)

    # Update Freq Plot
    l_H_mag.set_ydata(np.abs(H[pos_mask_padded]))

    fig.canvas.draw_idle()

s_dist.on_changed(update)
s_rho.on_changed(update)
s_q.on_changed(update)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
from scipy import signal

# --- 1. Signal Generation (Morse Code: "I T" -> .. -) ---
# "I" (..) "T" (-) is a good mix of short and long pulses to show ISI
fs = 1000  # Sampling frequency (Hz)
duration = 2.0
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

# Helper to create a pulse train
def generate_morse_signal(t):
    # Logic: 1 = High (5V), 0 = Low (0V)
    # Unit length for a "dot" in seconds
    dot_len = 0.08

    # Constructing a bitstream for "I T" (..  -) with pauses
    # Pattern: Dot, Space, Dot, Space(3x), Dash, Space
    # We map time to these bits
    sig = np.zeros_like(t)

    # Define pulses manually for clean control
    # Pulse 1 (Dot)
    p1_start, p1_end = 0.2, 0.2 + dot_len
    # Pulse 2 (Dot)
    p2_start, p2_end = p1_end + dot_len, p1_end + 2*dot_len
    # Pulse 3 (Dash - 3x length)
    p3_start, p3_end = p2_end + 3*dot_len, p2_end + 3*dot_len + 3*dot_len

    # Fill the signal array
    sig[(t >= p1_start) & (t < p1_end)] = 5
    sig[(t >= p2_start) & (t < p2_end)] = 5
    sig[(t >= p3_start) & (t < p3_end)] = 5

    return sig

sig_sender = generate_morse_signal(t)

# --- 2. The Physics (Cable Dispersion) ---
def apply_cable_dispersion(sig, distance_km):
    """
    Models the cable as a Low Pass Filter.
    Physics: As distance increases, the total Capacitance (C) and Resistance (R)
    increase linearly. The Cutoff Frequency (fc) is proportional to 1/RC.
    Therefore: fc is inversely proportional to Distance.
    """
    if distance_km <= 0.1:
        return sig # No dispersion

    # Arbitrary scaling constant for the simulation visual
    # In reality, fc = 1 / (2*pi * R_total * C_total)
    fc_base = 30 # Hz (Baseline bandwidth)
    fc = fc_base / (distance_km * 2)

    # Prevent numerical instability if fc is too close to 0
    fc = max(fc, 0.1)

    # Normalize for Nyquist (fs/2)
    w = fc / (fs / 2)

    # 1st Order Butterworth = Standard RC Circuit Step Response
    b, a = signal.butter(N=1, Wn=w, btype='low')

    # Apply filter
    sig_receiver = signal.lfilter(b, a, sig)
    return sig_receiver

# --- 3. Plotting & Interactivity ---
fig, ax = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(bottom=0.25)

# Initial Plot
d_init = 1.0
l_sender, = ax.plot(t, sig_sender, 'k--', alpha=0.5, label='Sender (Ideal)')
l_receiver, = ax.plot(t, apply_cable_dispersion(sig_sender, d_init), 'r-', linewidth=2, label='Receiver (Dispersed)')

# Aesthetics
ax.set_title("Telegraph Dispersion Simulator: Visualizing ISI", fontsize=14)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (V)")
ax.set_ylim(-1, 6)
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3)

# Add Threshold Line (The Logic Level)
ax.axhline(2.5, color='gray', linestyle=':', label='Logic Threshold (2.5V)')

# Slider setup
ax_slider = plt.axes([0.2, 0.1, 0.6, 0.03], facecolor='lightgoldenrodyellow')
slider = Slider(ax_slider, 'Cable Length (km)', 0.1, 10.0, valinit=d_init)

def update(val):
    d = slider.val
    # Recalculate filtered signal
    new_y = apply_cable_dispersion(sig_sender, d)
    l_receiver.set_ydata(new_y)
    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()